<a href="https://colab.research.google.com/github/utonez/belle/blob/main/HW3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import numpy as np

# ==========================================================
# 1. LAYER DEFINITIONS (The building blocks of the graph)
# ==========================================================

def affine_forward(x, w, b):
    """Computes the forward pass for an affine (fully-connected) layer."""
    out = x.reshape(x.shape[0], -1).dot(w) + b
    cache = (x, w, b)
    return out, cache

def affine_backward(dout, cache):
    """Computes the backward pass for an affine layer."""
    x, w, b = cache
    dx = dout.dot(w.T).reshape(x.shape)
    dw = x.reshape(x.shape[0], -1).T.dot(dout)
    db = np.sum(dout, axis=0)
    return dx, dw, db

def relu_forward(x):
    """Computes the forward pass for a layer of rectified linear units (ReLUs)."""
    out = np.maximum(0, x)
    cache = x
    return out, cache

def relu_backward(dout, cache):
    """Computes the backward pass for a layer of rectified linear units (ReLUs)."""
    x = cache
    dx = dout * (x > 0)
    return dx

def affine_relu_forward(x, w, b):
    """Convenience layer that performs an affine transform followed by a ReLU."""
    a, fc_cache = affine_forward(x, w, b)
    out, relu_cache = relu_forward(a)
    cache = (fc_cache, relu_cache)
    return out, cache

def affine_relu_backward(dout, cache):
    """Backward pass for the affine-relu convenience layer."""
    fc_cache, relu_cache = cache
    da = relu_backward(dout, relu_cache)
    dx, dw, db = affine_backward(da, fc_cache)
    return dx, dw, db

def softmax_loss(x, y):
    """Computes the loss and gradient for softmax classification."""
    shifted_logits = x - np.max(x, axis=1, keepdims=True)
    Z = np.sum(np.exp(shifted_logits), axis=1, keepdims=True)
    log_probs = shifted_logits - np.log(Z)
    probs = np.exp(log_probs)
    N = x.shape[0]
    loss = -np.sum(log_probs[np.arange(N), y]) / N
    dx = probs.copy()
    dx[np.arange(N), y] -= 1
    dx /= N
    return loss, dx

# ==========================================================
# 2. THE NETWORK CLASS
# ==========================================================

class TwoLayerNet(object):
    def __init__(self, input_dim=2, hidden_dim=2, num_classes=2, weight_scale=1.0):
        self.params = {}
        # Initializing weights randomly as per the 40-trial requirement
        self.params['W1'] = weight_scale * np.random.randn(input_dim, hidden_dim)
        self.params['b1'] = np.zeros(hidden_dim)
        self.params['W2'] = weight_scale * np.random.randn(hidden_dim, num_classes)
        self.params['b2'] = np.zeros(num_classes)

    def loss(self, X, y=None):
        W1, b1 = self.params['W1'], self.params['b1']
        W2, b2 = self.params['W2'], self.params['b2']

        # Forward pass: Affine -> ReLU -> Affine
        h1, cache1 = affine_relu_forward(X, W1, b1)
        scores, cache2 = affine_forward(h1, W2, b2)

        if y is None:
            return scores

        # Backward pass: Compute gradients using the Chain Rule
        loss, dscores = softmax_loss(scores, y)
        dh1, dW2, db2 = affine_backward(dscores, cache2)
        dx, dW1, db1 = affine_relu_backward(dh1, cache1)

        grads = {'W1': dW1, 'b1': db1, 'W2': dW2, 'b2': db2}
        return loss, grads

# ==========================================================
# 3. THE EXPERIMENT (40 TRIALS)
# ==========================================================

if __name__ == "__main__":
    # XOR Data
    X_train = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
    y_train = np.array([0, 1, 1, 0])

    num_trials = 40
    learning_rate = 1.0
    epochs = 3000
    successes = 0

    print(f"Running {num_trials} trials to solve XOR...")

    for i in range(num_trials):
        # Every trial starts with a fresh random initialization
        model = TwoLayerNet(input_dim=2, hidden_dim=2, num_classes=2, weight_scale=1.0)

        for ep in range(epochs):
            loss, grads = model.loss(X_train, y_train)
            for p in model.params:
                model.params[p] -= learning_rate * grads[p]

        # Evaluate: argmax converts the scores to class 0 or 1
        scores = model.loss(X_train)
        predictions = np.argmax(scores, axis=1)

        if np.array_equal(predictions, y_train):
            successes += 1
            print(f"Trial {i+1:2d}: SUCCESS")
        else:
            print(f"Trial {i+1:2d}: FAILED (Stuck in Local Minimum)")

    print("-" * 30)
    print(f"Final Result: {successes}/{num_trials} successes.")


Running 40 trials to solve XOR...
Trial  1: SUCCESS
Trial  2: FAILED (Stuck in Local Minimum)
Trial  3: SUCCESS
Trial  4: FAILED (Stuck in Local Minimum)
Trial  5: FAILED (Stuck in Local Minimum)
Trial  6: SUCCESS
Trial  7: FAILED (Stuck in Local Minimum)
Trial  8: FAILED (Stuck in Local Minimum)
Trial  9: FAILED (Stuck in Local Minimum)
Trial 10: FAILED (Stuck in Local Minimum)
Trial 11: FAILED (Stuck in Local Minimum)
Trial 12: FAILED (Stuck in Local Minimum)
Trial 13: FAILED (Stuck in Local Minimum)
Trial 14: FAILED (Stuck in Local Minimum)
Trial 15: FAILED (Stuck in Local Minimum)
Trial 16: FAILED (Stuck in Local Minimum)
Trial 17: SUCCESS
Trial 18: FAILED (Stuck in Local Minimum)
Trial 19: FAILED (Stuck in Local Minimum)
Trial 20: SUCCESS
Trial 21: FAILED (Stuck in Local Minimum)
Trial 22: FAILED (Stuck in Local Minimum)
Trial 23: FAILED (Stuck in Local Minimum)
Trial 24: SUCCESS
Trial 25: FAILED (Stuck in Local Minimum)
Trial 26: FAILED (Stuck in Local Minimum)
Trial 27: SUCCESS
